In [1]:
### PLEASE NOTICE THAT THIS ANALYSIS IS UNRELATED WITH ...

In [2]:
import os
import csv
import tqdm

In [3]:
def extract_score_from_sdf(filepath: str) -> tuple[str, str | None]:
    """
    Extracts the ENERGY value from the <Uni-Dock RESULT> block in an SDF file.
    """
    try:
        with open(filepath, 'r') as file:
            for line in file:
                if line.strip() == "> <Uni-Dock RESULT>":
                    result_line = next(file).strip()
                    if "ENERGY=" in result_line:
                        return os.path.basename(filepath), result_line.split("ENERGY=")[1].split()[0]
    except Exception:
        pass
    return os.path.basename(filepath), None

def generate_report(directory: str, output_csv: str = "scores.csv", score_field: str = "score"):
    """
    Processes all .sdf files in a directory, extracts score values in parallel,
    and writes the results as a CSV: filename,score.
    """

    # Enumerate SDF files
    sdf_files = [os.path.join(directory, f) for f in sorted(os.listdir(directory)) if f.endswith(".sdf")]

    # Get results
    results = [extract_score_from_sdf(file) for file in tqdm.tqdm(sdf_files)]

    # Write results
    with open(output_csv, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["compound", "score"])
        for compound, score in results:
            writer.writerow([compound.replace("_out.sdf", ""), score if score is not None else ""])

In [4]:
PATH = "/home/acomajuncosa/Documents_GPU/docking-unidock_benchmarking"
MODES = ['fast', 'balance', 'detail']

for mode in MODES:

    generate_report(os.path.join(PATH, f"output_{mode}"), os.path.join(PATH, f"report_{mode}.csv"))

100%|██████████| 100151/100151 [00:17<00:00, 5605.84it/s]
